# Redshift-Space Distortion (RSD) Direct Estimator

This notebook mirrors the weighted RSD subvolume workflow, but it uses a direct periodic-box estimator instead of Landy-Szalay.

The key difference is that the random catalog is replaced by the analytic RR expectation for a uniform periodic cube, so the estimator is effectively:

$$\xi(s,\mu) = \frac{DD(s,\mu)}{RR_{\mathrm{analytic}}(s,\mu)} - 1$$

The subvolume weighting still matters, but only for correcting the missing auto/cross pair structure caused by subsampling.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Add the GALFORM python source package
sys.path.append(str(Path("../../src").resolve()))

from analysis.redshift_space_distortions.subvol_weighted_multipoles import (
    compute_direct_rsd_multipoles,
    compute_weighted_direct_rsd_multipoles,
 )
from utils.read_galaxies import read_galaxy_arrays
from utils import setconfig
from utils.stats import positive_se, positive_std, positive_percentile
from config import Cosmology, get_snapshot_redshift

setconfig()
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 130
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Load Real GALFORM Galaxies

We use the same GALFORM snapshot family as the LS notebook, but the estimator itself is periodic-box analytic, so no random catalog is needed.

The line-of-sight shift is still built from `vzgal`:

$$\Delta s_{\parallel} \approx v_z \; h / H(z)$$

In [ ]:
BASE_DIR = Path("/cosma5/data/durham/dc-hick2/Galform_Out/L800/lc16")
IZ = 155
BOXSIZE = 542.16
K_TOTAL = 1024

MAX_GAL_PER_SUBVOL = 3000
MHALO_MIN = 1e10
CENTRALS_ONLY = False
IVOLS_FOR_COMPARISON = [0, 1, 2]
REFERENCE_IVOLS = [0, 1, 2]

z_snap = get_snapshot_redshift(f"iz{IZ}")
if z_snap is None:
    z_snap = 0.0

h = Cosmology.h
Ez = np.sqrt(Cosmology.OMEGA_M * (1.0 + z_snap) ** 3 + Cosmology.OMEGA_L)
H_z = 100.0 * h * Ez  # km/s/Mpc

print(f"Using {BASE_DIR}/iz{IZ}")
print(f"Snapshot redshift z={z_snap:.3f}, H(z)={H_z:.2f} km/s/Mpc")


def load_rsd_positions_for_ivols(ivols):
    pos_chunks = []
    label_chunks = []

    for label, ivol in enumerate(ivols):
        arrays, _ = read_galaxy_arrays(
            iz_path=str(BASE_DIR / f"iz{IZ}"),
            ivol=int(ivol),
            fields=["vzgal"],
            include_positions=True,
            include_derived=True,
            centrals_only=CENTRALS_ONLY,
            mhalo_min=MHALO_MIN,
        )

        x = arrays["x"]
        y = arrays["y"]
        z = arrays["z"]
        vz = arrays["vzgal"]

        n = len(x)
        if n > MAX_GAL_PER_SUBVOL:
            idx = np.random.choice(n, size=MAX_GAL_PER_SUBVOL, replace=False)
            x, y, z, vz = x[idx], y[idx], z[idx], vz[idx]

        ds_par = (vz / H_z) * h  # Mpc/h
        z_rsd = (z + ds_par) % BOXSIZE

        pos = np.column_stack([x, y, z_rsd])
        labels = np.full(pos.shape[0], label, dtype=np.int64)

        pos_chunks.append(pos)
        label_chunks.append(labels)

        print(f"ivol{ivol}: loaded {pos.shape[0]} galaxies")

    gal_pos = np.vstack(pos_chunks)
    gal_labels = np.concatenate(label_chunks)
    return gal_pos, gal_labels


s_bins = np.logspace(-1, 1.4, 18)
mu_max = 1.0
n_mu_bins = 16

reference_pos, reference_labels = load_rsd_positions_for_ivols(REFERENCE_IVOLS)
reference_corrected = compute_weighted_direct_rsd_multipoles(
    galaxy_pos=reference_pos,
    galaxy_labels=reference_labels,
    s_bins=s_bins,
    mu_max=mu_max,
    n_mu_bins=n_mu_bins,
    k_total=K_TOTAL,
    boxsize=BOXSIZE,
    nthreads=4,
)

print(f"Reference direct estimator loaded {reference_pos.shape[0]} galaxies")

## 2. Compute Direct RSD Multipoles For 1, 2, and 3 Subvolumes

For each $m \in \{1,2,3\}$ we compute:
- the unweighted direct periodic estimator
- the subvolume-weighted direct estimator

The largest available case ($m=3$ here) acts as the corrected direct-estimator reference for the difference plots.

In [ ]:
results_by_m = {}

for m in [1, 2, 3]:
    ivols = IVOLS_FOR_COMPARISON[:m]
    gal_pos, labels = load_rsd_positions_for_ivols(ivols)

    standard = compute_direct_rsd_multipoles(
        galaxy_pos=gal_pos,
        s_bins=s_bins,
        mu_max=mu_max,
        n_mu_bins=n_mu_bins,
        boxsize=BOXSIZE,
        nthreads=4,
    )

    corrected = compute_weighted_direct_rsd_multipoles(
        galaxy_pos=gal_pos,
        galaxy_labels=labels,
        s_bins=s_bins,
        mu_max=mu_max,
        n_mu_bins=n_mu_bins,
        k_total=K_TOTAL,
        boxsize=BOXSIZE,
        nthreads=4,
    )

    results_by_m[m] = {"standard": standard, "corrected": corrected}
    print(
        f"m={m}: ngal={gal_pos.shape[0]}, standard_xi0_first={standard['xi0'][0]:.4e}, "
        f"corrected_alpha={corrected['alpha']:.6f}, corrected_beta={corrected['beta'] if np.isfinite(corrected['beta']) else np.nan}"
    )

## 3. Direct RSD Curves and Differences

This section uses the weighted direct estimator and compares each subset to the corrected direct-estimator reference built from all available loaded subvolumes.

Figures:
- raw $s^2\xi_\ell(s)$ curves
- absolute difference from the reference
- percentage difference from the reference

In [ ]:
PLOT_ROOT = Path("./_plots/rsd_weighted_correction_direct")
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

def aligned_s2(values, s):
    return (s ** 2) * values

eps = 1e-12
plot_outputs = []

fig_raw, axes_raw = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
fig_abs, axes_abs = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
fig_pct, axes_pct = plt.subplots(2, 2, figsize=(14, 10), sharex=True)

mode_layout = [("standard", 0), ("corrected", 1)]
reference_s = reference_corrected["s"]
reference_xi0 = reference_corrected["xi0"]
reference_xi2 = reference_corrected["xi2"]

for mode, col in mode_layout:
    ax_raw_0 = axes_raw[0, col]
    ax_raw_2 = axes_raw[1, col]
    ax_abs_0 = axes_abs[0, col]
    ax_abs_2 = axes_abs[1, col]
    ax_pct_0 = axes_pct[0, col]
    ax_pct_2 = axes_pct[1, col]

    for m in [1, 2, 3]:
        res = results_by_m[m][mode]
        s = res["s"]
        xi0 = res["xi0"]
        xi2 = res["xi2"]
        color = plt.cm.viridis(np.linspace(0.15, 0.95, 3))[m - 1]
        label = f"m={m}"

        ax_raw_0.plot(s, aligned_s2(xi0, s), lw=2, color=color, label=label)
        ax_raw_2.plot(s, aligned_s2(xi2, s), lw=2, color=color, label=label)

        ref0 = np.interp(s, reference_s, reference_xi0)
        ref2 = np.interp(s, reference_s, reference_xi2)

        ax_abs_0.plot(s, np.abs(aligned_s2(xi0, s) - aligned_s2(ref0, s)), lw=2, color=color, label=label)
        ax_abs_2.plot(s, np.abs(aligned_s2(xi2, s) - aligned_s2(ref2, s)), lw=2, color=color, label=label)

        pct0 = 100.0 * (xi0 - ref0) / (np.abs(ref0) + eps)
        pct2 = 100.0 * (xi2 - ref2) / (np.abs(ref2) + eps)
        ax_pct_0.plot(s, pct0, lw=2, color=color, label=label)
        ax_pct_2.plot(s, pct2, lw=2, color=color, label=label)

    ax_raw_0.plot(reference_s, aligned_s2(reference_xi0, reference_s), color="black", lw=3, linestyle="--", label="reference")
    ax_raw_2.plot(reference_s, aligned_s2(reference_xi2, reference_s), color="black", lw=3, linestyle="--", label="reference")

    ax_raw_0.set_xscale("log")
    ax_raw_2.set_xscale("log")
    ax_abs_0.set_xscale("log")
    ax_abs_2.set_xscale("log")
    ax_pct_0.set_xscale("log")
    ax_pct_2.set_xscale("log")

    ax_raw_0.set_yscale("log")
    ax_raw_2.set_yscale("symlog", linthresh=10.0)

    ax_raw_0.set_title(f"{mode.title()} monopole")
    ax_raw_2.set_title(f"{mode.title()} quadrupole")
    ax_abs_0.set_title(f"{mode.title()} monopole |run - ref|")
    ax_abs_2.set_title(f"{mode.title()} quadrupole |run - ref|")
    ax_pct_0.set_title(f"{mode.title()} monopole % diff")
    ax_pct_2.set_title(f"{mode.title()} quadrupole % diff")

    ax_pct_0.axhline(0.0, color="black", lw=1.5, linestyle="--")
    ax_pct_2.axhline(0.0, color="black", lw=1.5, linestyle="--")

axes_raw[0, 0].set_ylabel(r"$s^2\,\xi_0(s)$")
axes_raw[1, 0].set_ylabel(r"$s^2\,\xi_2(s)$")
axes_raw[1, 0].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")
axes_raw[1, 1].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")

axes_abs[0, 0].set_ylabel(r"$|\Delta(s^2\xi_0)|$")
axes_abs[1, 0].set_ylabel(r"$|\Delta(s^2\xi_2)|$")
axes_abs[1, 0].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")
axes_abs[1, 1].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")

axes_pct[0, 0].set_ylabel(r"$100\,\Delta/(|ref|+\epsilon)$")
axes_pct[1, 0].set_ylabel(r"$100\,\Delta/(|ref|+\epsilon)$")
axes_pct[1, 0].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")
axes_pct[1, 1].set_xlabel(r"$s\,[h^{-1}\mathrm{Mpc}]$")

for ax in [axes_raw[0, 1], axes_raw[1, 1], axes_abs[0, 1], axes_abs[1, 1], axes_pct[0, 1], axes_pct[1, 1]]:
    ax.legend(fontsize=8, ncol=2)

fig_raw.suptitle("Direct RSD curves by subvolume count", y=1.01)
fig_abs.suptitle("Direct RSD absolute difference to reference", y=1.01)
fig_pct.suptitle("Direct RSD percentage difference to reference", y=1.01)

for fig, tag in [(fig_raw, "curves"), (fig_abs, "absdiff"), (fig_pct, "pctdiff")]:
    fig.tight_layout()
    outpath = PLOT_ROOT / f"rsd_direct_{tag}.png"
    fig.savefig(outpath, bbox_inches="tight")
    plot_outputs.append(outpath)

display(fig_raw)
display(fig_abs)
display(fig_pct)

print(f"Saved {len(plot_outputs)} figure files to {PLOT_ROOT}")

## 4. Convergence Table

We summarise the direct-estimator differences against the reference direct estimate from the largest loaded sample. This keeps the notebook fully analytic and avoids any random-catalog dependence.

In [ ]:
convergence_rows = []
eps = 1e-12

for m in [1, 2, 3]:
    for mode in ["standard", "corrected"]:
        res = results_by_m[m][mode]
        s = res["s"]
        ref0 = np.interp(s, reference_corrected["s"], reference_corrected["xi0"])
        ref2 = np.interp(s, reference_corrected["s"], reference_corrected["xi2"])

        convergence_rows.append({
            "m": m,
            "mode": mode,
            "med_abs_diff_s2xi0": float(np.nanmedian(np.abs((s ** 2) * (res["xi0"] - ref0)))),
            "med_abs_diff_s2xi2": float(np.nanmedian(np.abs((s ** 2) * (res["xi2"] - ref2)))),
            "med_abs_pct_diff_s2xi0": float(np.nanmedian(np.abs(100.0 * (res["xi0"] - ref0) / (np.abs(ref0) + eps)))),
            "med_abs_pct_diff_s2xi2": float(np.nanmedian(np.abs(100.0 * (res["xi2"] - ref2) / (np.abs(ref2) + eps)))),
        })

convergence_df = pd.DataFrame(convergence_rows).sort_values(["mode", "m"])
display(convergence_df)

conv_path = PLOT_ROOT / "rsd_direct_convergence_metrics.csv"
convergence_df.to_csv(conv_path, index=False)
print(f"Saved convergence table: {conv_path}")

## 5. Load Convergence Study Results

Once the batch job submission completes (110 jobs across iz155 and iz63), this cell loads all the CSV results and prepares them for convergence analysis.


In [ ]:
import glob
from collections import defaultdict

# Load all CSV results from completed convergence study jobs
DATA_ROOT = Path("../../data/redshift_space_distortions/fullbox_direct/lc16/L800")
PLOT_ROOT = Path("../../data/redshift_space_distortions/plots")
PLOT_ROOT.mkdir(parents=True, exist_ok=True)
SNAPSHOTS = [155, 207]  # Updated: iz63 not available, using iz207 instead

convergence_data = {}  # {iz: {n_subvol: [list of dataframes from multiple runs]}}

for iz in SNAPSHOTS:
    iz_dir = DATA_ROOT / f"iz{iz}"
    convergence_data[iz] = defaultdict(list)
    
    if not iz_dir.exists():
        print(f"Warning: Directory not found: {iz_dir}")
        continue
    
    # Find all CSV files matching the pattern: rsd_fullbox_weighted_direct_*_iz{iz}_n{n}_r{run_id}.csv
    pattern = str(iz_dir / f"rsd_fullbox_weighted_direct_*_iz{iz}_n*_r*.csv")
    csv_files = sorted(glob.glob(pattern))
    
    if not csv_files:
        print(f"No CSV files found for iz{iz} in {iz_dir}")
        continue
    
    print(f"\niz{iz}: Found {len(csv_files)} CSV files")
    
    # Parse filename to extract n_subvol (normalize 'fullbox' to 1024)
    for csv_file in csv_files:
        try:
            fname = Path(csv_file).stem  # e.g., rsd_fullbox_weighted_direct_L800_lc16_iz155_n1_r0
            parts = fname.split('_')
            n_subvol = None
            n_str = [p for p in parts if p.startswith('n')]
            if n_str:
                try:
                    n_subvol = int(n_str[0][1:])  # extract n value
                except ValueError:
                    n_subvol = None
            # Fallback: if filename contains 'fullbox' token but no n, treat as full box (1024)
            if n_subvol is None:
                if 'fullbox' in parts or 'fullbox' in fname or 'full' in parts:
                    n_subvol = 1024
            if n_subvol is not None:
                df = pd.read_csv(csv_file)
                convergence_data[iz][int(n_subvol)].append(df)
        except Exception as e:
            print(f"  Error loading {csv_file}: {e}")

# Print summary
print("\n" + "="*60)
print("CONVERGENCE DATA SUMMARY")
print("="*60)
for iz in SNAPSHOTS:
    if iz in convergence_data and convergence_data[iz]:
        n_values = sorted(convergence_data[iz].keys())
        print(f"\niz{iz}:")
        for n in n_values:
            n_runs = len(convergence_data[iz][n])
            label = "fullbox (n=1024)" if n == 1024 else f"n_subvol={n:4d}"
            print(f"  {label}: {n_runs} run(s)")
    else:
        print(f"\niz{iz}: No data loaded")

print("="*60)

## 6. Convergence Plots: Monopole and Quadrupole

Plot how the monopole ($\xi_0$) and quadrupole ($\xi_2$) converge as a function of $n_{\rm subvol}$.

For each snapshot, we show:
- **Mean multipoles** across all runs at a given $n_{\rm subvol}$
- **Error bars** (std dev) when multiple runs exist
- **Color progression** from small (blue) to large (yellow) $n_{\rm subvol}$
- **Reference level** at $n=1024$ (full box) as horizontal dashed lines


In [ ]:
# Compute convergence statistics for each n_subvol and snapshot
convergence_stats = {}

for iz in SNAPSHOTS:
    convergence_stats[iz] = {}
    
    if iz not in convergence_data or not convergence_data[iz]:
        print(f"Skipping iz{iz}: no data available")
        continue
    
    for n_subvol in sorted(convergence_data[iz].keys()):
        dfs = convergence_data[iz][n_subvol]
        
        if not dfs:
            continue
        
        # Stack all runs
        s_ref = dfs[0]["s"].values
        xi0_array = np.array([df["xi0"].values for df in dfs])
        xi2_array = np.array([df["xi2"].values for df in dfs])
        
        n_runs = len(dfs)
        convergence_stats[iz][n_subvol] = {
            "s": s_ref,
            "xi0_mean": np.mean(xi0_array, axis=0),
            "xi0_se": positive_se(xi0_array, axis=0) if n_runs > 1 else np.zeros_like(s_ref),
            "xi2_mean": np.mean(xi2_array, axis=0),
            "xi2_se": positive_se(xi2_array, axis=0) if n_runs > 1 else np.zeros_like(s_ref),
            "n_runs": n_runs,
        }
        

## 7. Convergence Metrics: Difference from Reference

Show how the median absolute percentage difference converges as a function of $n_{\rm subvol}$.


In [ ]:
# Compute and plot convergence metrics
for iz in SNAPSHOTS:
    if iz not in convergence_stats or not convergence_stats[iz]:
        continue
    
    n_values = sorted(convergence_stats[iz].keys())
    
    # Get reference (n=1024)
    if 1024 not in convergence_stats[iz]:
        print(f"Skipping metrics for iz{iz}: n=1024 reference not available")
        continue
    
    ref_data = convergence_stats[iz][1024]
    ref_s = ref_data["s"]
    ref_xi0 = ref_data["xi0_mean"]
    ref_xi2 = ref_data["xi2_mean"]
    
    eps = 1e-12
    
    # Compute metrics for each n_subvol
    metrics = []
    for n_subvol in n_values:
        data = convergence_stats[iz][n_subvol]
        s = data["s"]
        xi0 = data["xi0_mean"]
        xi2 = data["xi2_mean"]
        
        # Interpolate reference to match sample s
        ref_xi0_interp = np.interp(s, ref_s, ref_xi0)
        ref_xi2_interp = np.interp(s, ref_s, ref_xi2)
        
        # Compute median absolute percentage differences
        pct_diff_xi0 = np.abs(100.0 * (xi0 - ref_xi0_interp) / (np.abs(ref_xi0_interp) + eps))
        pct_diff_xi2 = np.abs(100.0 * (xi2 - ref_xi2_interp) / (np.abs(ref_xi2_interp) + eps))
        
        med_pct_diff_xi0 = np.nanmedian(pct_diff_xi0)
        med_pct_diff_xi2 = np.nanmedian(pct_diff_xi2)
        
        metrics.append({
            "n_subvol": n_subvol,
            "n_runs": data["n_runs"],
            "med_pct_diff_xi0": med_pct_diff_xi0,
            "med_pct_diff_xi2": med_pct_diff_xi2,
        })
    
    metrics_df = pd.DataFrame(metrics)
    
    # Plot convergence rates
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    # Monopole convergence
    axes[0].loglog(
        metrics_df["n_subvol"],
        metrics_df["med_pct_diff_xi0"],
        "o-",
        lw=2.5,
        markersize=8,
        color="steelblue",
        label="$|\Delta \\xi_0| / |\\xi_0|$ (median)",
    )
    axes[0].set_xlabel(r"$n_{\rm subvol}$", fontsize=12)
    axes[0].set_ylabel(r"Median $|\Delta \xi_0 / \xi_0|$ (%)", fontsize=12)
    axes[0].set_title(f"Monopole Convergence Rate: iz{iz}", fontsize=13, fontweight="bold")
    axes[0].grid(True, alpha=0.3, which="both")
    axes[0].legend(fontsize=10)
    
    # Quadrupole convergence
    axes[1].loglog(
        metrics_df["n_subvol"],
        metrics_df["med_pct_diff_xi2"],
        "o-",
        lw=2.5,
        markersize=8,
        color="coral",
        label="$|\Delta \\xi_2| / |\\xi_2|$ (median)",
    )
    axes[1].set_xlabel(r"$n_{\rm subvol}$", fontsize=12)
    axes[1].set_ylabel(r"Median $|\Delta \xi_2 / \xi_2|$ (%)", fontsize=12)
    axes[1].set_title(f"Quadrupole Convergence Rate: iz{iz}", fontsize=13, fontweight="bold")
    axes[1].grid(True, alpha=0.3, which="both")
    axes[1].legend(fontsize=10)
    
    fig.suptitle(f"Convergence vs. Full Box (n=1024): L800/lc16 (iz{iz})", fontsize=14, fontweight="bold", y=1.00)
    fig.tight_layout()
    
    outpath = PLOT_ROOT / f"convergence_metrics_iz{iz}.png"
    fig.savefig(outpath, dpi=150, bbox_inches="tight")
    print(f"\nSaved: {outpath}\n")
    
    # Print metrics table
    print(f"Convergence Metrics (iz{iz}):")
    print(metrics_df.to_string(index=False))
    print()
    
    display(fig)
    plt.close(fig)
